<a href="https://colab.research.google.com/github/steveonyeke/python-ai-governance/blob/main/project-2-llm-evaluation-suite/06a_langfuse_custom_scores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 6a: Wiring RAGAS, DeepEval, and Promptfoo Scores into Langfuse as Custom Scores

**Goal:** Wire scores from Phases 2 through 5c into Langfuse as structured custom scores attached to traced runs, building the production observability layer described in the Talabat prep's "drift monitoring = Langfuse Phase 6" pillar.

**Design decision, built to be flip-ready correctly, not just superficially:** this notebook does not generate its own scores, it wires in scores already produced upstream. Flipping `SIMULATED_OUTPUT` to `False` here is therefore not sufficient by itself to make this notebook "real": each upstream phase's saved results file carries its own `"simulated"` field, and this notebook checks that field per source before pushing anything to a live Langfuse dashboard as genuine. A score from a still-simulated upstream phase is pushed to Langfuse tagged as simulated, never silently presented as real just because this notebook's own flag was flipped.

**Tools:** Langfuse v4, reads saved JSON output from Phases 2a/2b, 3a/3b, 4a/4b, 5a/5b/5c

**Date:** July 2026

**Status:** In progress. Wiring logic is written to run for real the moment Langfuse credentials exist and is not blocked on Gemini or Claude billing at all, since this phase moves already-computed numbers, it makes no model calls of its own.

In [2]:
# Cell 2: Mount Drive and load all upstream results
# Corrected filenames, checked against each notebook's actual save cell
# rather than guessed.

from google.colab import drive
drive.mount('/content/drive')

import os, json

DRIVE_PATH = "/content/drive/MyDrive/python-ai-governance-p2/data/"

UPSTREAM_SOURCES = {
    "phase02a": "phase02a_gemini_judge_results.json",
    "phase02b": "phase02b_claude_judge_results.json",
    "phase03a": "phase03a_deepeval_rag_results.json",
    "phase03b": "phase03b_governance_metrics_results.json",
    "phase04a": "phase04a_aspect_critic_results.json",
    "phase04b": "phase04b_judge_alignment_results.json",
    "phase05a": "phase05a_promptfoo_owasp_results.json",
    "phase05b": "phase05b_promptfoo_owasp_agentic_results.json",
    "phase05c": "phase05c_mitre_atlas_mapping_results.json",
}

upstream_data = {}
upstream_status = {}

for phase_key, filename in UPSTREAM_SOURCES.items():
    path = DRIVE_PATH + filename
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        upstream_data[phase_key] = data
        is_simulated = data.get("simulated", None)
        upstream_status[phase_key] = is_simulated
    else:
        upstream_data[phase_key] = None
        upstream_status[phase_key] = "MISSING"

print("UPSTREAM SOURCE STATUS")
print("=" * 60)
for phase_key, status in upstream_status.items():
    if status == "MISSING":
        flag = "MISSING FILE"
    elif status is True:
        flag = "SIMULATED"
    elif status is False:
        flag = "REAL"
    else:
        flag = "NO 'simulated' FIELD, mixed real/simulated by design (05c only)"
    print(f"  {phase_key}: {flag}")

real_count = sum(1 for v in upstream_status.values() if v is False)
simulated_count = sum(1 for v in upstream_status.values() if v is True)
print()
print(f"Real: {real_count} | Simulated: {simulated_count} | "
      f"Missing/mixed: {len(upstream_status) - real_count - simulated_count}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
UPSTREAM SOURCE STATUS
  phase02a: SIMULATED
  phase02b: SIMULATED
  phase03a: SIMULATED
  phase03b: SIMULATED
  phase04a: SIMULATED
  phase04b: SIMULATED
  phase05a: SIMULATED
  phase05b: SIMULATED
  phase05c: NO 'simulated' FIELD, mixed real/simulated by design (05c only)

Real: 0 | Simulated: 8 | Missing/mixed: 1


In [3]:
# Cell 3: Install packages

!pip install langfuse pandas --quiet

print("Packages installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 669.4/669.4 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 17.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.4.0 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.4.0 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.
Packages installed.


In [4]:
# Cell 4: Simulated output flag and Langfuse client
# Unlike Phases 2-5, this notebook's real branch does not depend on Gemini
# or Claude billing at all, it only needs Langfuse credentials, which are
# independent of model API costs. This means 06a's own "real mode" can be
# switched on today if Langfuse keys exist, even while every upstream
# phase's scores remain simulated. The two are genuinely decoupled, which
# is exactly why Cell 2's per-source check matters: this notebook being
# "real" and the data it's wiring in being "real" are two different facts.

SIMULATED_OUTPUT = True

from google.colab import userdata

if not SIMULATED_OUTPUT:
    from langfuse import Langfuse
    langfuse = Langfuse(
        public_key=userdata.get('LANGFUSE_PUBLIC_KEY'),
        secret_key=userdata.get('LANGFUSE_SECRET_KEY'),
        host="https://cloud.langfuse.com"
    )
    print("Langfuse client initialised. Ready to push real custom scores.")
else:
    print("[SIMULATED] Langfuse client not initialised.")
    print(f"SIMULATED_OUTPUT = {SIMULATED_OUTPUT}")

print()
print("Reminder from Cell 2: even when this flag is False, any individual")
print("upstream score is only pushed as 'real' if its source file's own")
print("'simulated' field says so. This flag alone does not launder")
print("simulated upstream data into looking real.")

[SIMULATED] Langfuse client not initialised.
SIMULATED_OUTPUT = True

Reminder from Cell 2: even when this flag is False, any individual
upstream score is only pushed as 'real' if its source file's own
'simulated' field says so. This flag alone does not launder
simulated upstream data into looking real.


In [5]:
# Cell 5: Extract a consistent score summary from each upstream file
# Every phase saved a different JSON shape (verified against each notebook's
# actual save cell, not assumed). This normalizes each into one consistent
# format before pushing to Langfuse, rather than writing nine different
# ad hoc pushes.

def extract_02a(data):
    return {
        "headline_label": "RAGAS faithfulness (Gemini judge, same-family)",
        "headline_score": data["aggregate_scores"]["faithfulness"]["mean"],
        "detail": {k: v["mean"] for k, v in data["aggregate_scores"].items()},
    }

def extract_02b(data):
    return {
        "headline_label": "Same-family quality inflation (Gemini vs Claude judge)",
        "headline_score": data["bias_measurement"]["average_quality_inflation"],
        "detail": {
            "noise_detection_gap": data["bias_measurement"]["noise_detection_gap"],
            "aggregate_scores": {k: v["mean"] for k, v in data["aggregate_scores"].items()},
        },
    }

def extract_03a(data):
    matches_str = data["outcome_accuracy"]  # e.g. "5/5"
    matched, total = (int(x) for x in matches_str.split("/"))
    return {
        "headline_label": "DeepEval RAG suite outcome accuracy",
        "headline_score": matched / total,
        "detail": {"queue_summary": data["queue_summary"]},
    }

def extract_03b(data):
    matches_str = data["overall"]["outcome_accuracy"]
    matched, total = (int(x) for x in matches_str.split("/"))
    return {
        "headline_label": "DeepEval governance metrics outcome accuracy",
        "headline_score": matched / total,
        "detail": {"adversarial_detection_rate": data["overall"]["adversarial_detection_rate"]},
    }

def extract_04a(data):
    return {
        "headline_label": "RAGAS AspectCritic pre-calibration alignment",
        "headline_score": data["alignment"]["pre_calibration_score"],
        "detail": {"routing_summary": data["routing_summary"]},
    }

def extract_04b(data):
    return {
        "headline_label": "Judge alignment, post-calibration",
        "headline_score": data["aspect_versions"]["v2"]["alignment_score"],
        "detail": {},
    }

def extract_05a(data):
    return {
        "headline_label": "OWASP LLM Top 10 detection rate",
        "headline_score": data["detection_rate"],
        "detail": {"detected_count": data["detected_count"], "total": data["attack_case_count"]},
    }

def extract_05b(data):
    return {
        "headline_label": "OWASP Agentic Top 10 detection rate",
        "headline_score": data["detection_rate"],
        "detail": {"detected_count": data["detected_count"], "total": data["attack_case_count"]},
    }

def extract_05c(data):
    return {
        "headline_label": "ATLAS mapping direct-match coverage",
        "headline_score": data["mapping_confidence_breakdown"]["direct"] / data["mapping_confidence_breakdown"]["total_categories"],
        "detail": {"nist_rmf_report_card": data["nist_rmf_report_card"]},
    }

EXTRACTORS = {
    "phase02a": extract_02a, "phase02b": extract_02b,
    "phase03a": extract_03a, "phase03b": extract_03b,
    "phase04a": extract_04a, "phase04b": extract_04b,
    "phase05a": extract_05a, "phase05b": extract_05b,
    "phase05c": extract_05c,
}

normalized_scores = {}
for phase_key, data in upstream_data.items():
    if data is None:
        continue
    extracted = EXTRACTORS[phase_key](data)
    # 05c has no single "simulated" field, mixed by design, see Cell 2.
    is_simulated = data.get("simulated", "mixed_unresolved")
    normalized_scores[phase_key] = {**extracted, "simulated": is_simulated}

print("NORMALIZED SCORES")
print("=" * 70)
for phase_key, s in normalized_scores.items():
    sim_flag = "[SIMULATED]" if s["simulated"] is True else (
               "[MIXED]" if s["simulated"] == "mixed_unresolved" else "[REAL]")
    print(f"  {phase_key} {sim_flag}: {s['headline_label']} = {s['headline_score']:.3f}")

NORMALIZED SCORES
  phase02a [SIMULATED]: RAGAS faithfulness (Gemini judge, same-family) = 0.960
  phase02b [SIMULATED]: Same-family quality inflation (Gemini vs Claude judge) = 0.151
  phase03a [SIMULATED]: DeepEval RAG suite outcome accuracy = 1.000
  phase03b [SIMULATED]: DeepEval governance metrics outcome accuracy = 0.857
  phase04a [SIMULATED]: RAGAS AspectCritic pre-calibration alignment = 0.833
  phase04b [SIMULATED]: Judge alignment, post-calibration = 1.000
  phase05a [SIMULATED]: OWASP LLM Top 10 detection rate = 1.000
  phase05b [SIMULATED]: OWASP Agentic Top 10 detection rate = 1.000
  phase05c [MIXED]: ATLAS mapping direct-match coverage = 0.550


In [6]:
# Cell 6: Push normalized scores into Langfuse as custom scores
# Each score is tagged with its own source phase's real/simulated status,
# never inherited from this notebook's own SIMULATED_OUTPUT flag. This is
# the mechanism that makes this notebook honest even once it goes live.

def create_trace(name: str, metadata: dict) -> dict:
    trace = {"name": name, "metadata": metadata, "scores": []}
    if not SIMULATED_OUTPUT:
        lf_trace = langfuse.trace(name=name, metadata=metadata)
        trace["langfuse_id"] = lf_trace.id
    else:
        trace["langfuse_id"] = f"simulated-{name}"
    return trace


def log_score(trace: dict, name: str, value: float,
               comment: str = "", source_simulated=None) -> None:
    trace["scores"].append({
        "name": name, "value": round(value, 4), "comment": comment,
        "source_simulated": source_simulated,
    })
    # A score only ever gets pushed to a live Langfuse dashboard as genuine
    # if BOTH this notebook is in real mode AND its specific source phase
    # was itself real. Simulated source data never reaches a live dashboard
    # tagged as anything other than simulated, regardless of this
    # notebook's own flag.
    if not SIMULATED_OUTPUT and source_simulated is False:
        langfuse.score(trace_id=trace["langfuse_id"], name=name,
                        value=value, comment=comment)
    elif not SIMULATED_OUTPUT and source_simulated is not False:
        print(f"  [SKIPPED LIVE PUSH] {name}: source phase is not confirmed "
              f"real, would not push as genuine even though this notebook "
              f"is in real mode.")


traces_6a = []
for phase_key, s in normalized_scores.items():
    trace = create_trace(
        name=f"phase06a_{phase_key}_custom_score",
        metadata={
            "phase": "06a", "source_phase": phase_key,
            "source_simulated": s["simulated"], "notebook_simulated": SIMULATED_OUTPUT,
        }
    )
    log_score(trace, f"{phase_key}_headline_score", s["headline_score"],
              comment=s["headline_label"], source_simulated=s["simulated"])
    traces_6a.append(trace)

print(f"Traces created: {len(traces_6a)}")
print(f"All tagged with per-source simulated status, none inherited from "
      f"this notebook's own flag.")

Traces created: 9
All tagged with per-source simulated status, none inherited from this notebook's own flag.


In [7]:
# Cell 7: Save results to Drive

import json
from datetime import datetime

output_6a = {
    "phase": "06a_langfuse_custom_scores",
    "timestamp": datetime.now().isoformat(),
    "notebook_simulated": SIMULATED_OUTPUT,
    "upstream_source_status": {
        k: (v if not isinstance(v, bool) else v) for k, v in upstream_status.items()
    },
    "normalized_scores": normalized_scores,
    "trace_count": len(traces_6a),
    "design_notes": {
        "per_source_gating": (
            "This notebook's own SIMULATED_OUTPUT flag controls only whether "
            "it attempts a live Langfuse connection. Whether any individual "
            "score is pushed to that live dashboard as genuine depends "
            "entirely on its source phase's own recorded 'simulated' status, "
            "checked in Cell 2 and enforced again in Cell 6's log_score(). "
            "A future real run of this notebook against still-simulated "
            "upstream phases will connect to Langfuse for real but correctly "
            "skip pushing any of those nine scores as genuine, printing an "
            "explicit skip notice for each one instead."
        ),
        "current_status": (
            "All 9 upstream phases are currently simulated (or mixed, in "
            "05c's case). 0 real scores exist anywhere in Project 2 as of "
            "this run. This notebook's wiring logic is real and ready; "
            "there is nothing real yet for it to wire in."
        )
    }
}

output_path = DRIVE_PATH + "phase06a_langfuse_custom_scores_results.json"
with open(output_path, "w") as f:
    json.dump(output_6a, f, indent=2)

print(f"Results saved: {output_path}")
print()
print("Summary:")
print(f"  Upstream sources found: {sum(1 for v in upstream_data.values() if v is not None)}/9")
print(f"  Real: 0 | Simulated: 8 | Mixed: 1")
print(f"  Traces wired: {len(traces_6a)}")

Results saved: /content/drive/MyDrive/python-ai-governance-p2/data/phase06a_langfuse_custom_scores_results.json

Summary:
  Upstream sources found: 9/9
  Real: 0 | Simulated: 8 | Mixed: 1
  Traces wired: 9


## Phase 6a Findings: Wiring RAGAS, DeepEval, and Promptfoo Scores into Langfuse

**Tooling:** Langfuse v4, reads saved results from Phases 2a through 5c

**What was built:** A per-source real/simulated status check across all nine upstream results files (Phase 2a through 5c), a normalization layer extracting a consistent headline score from each phase's genuinely different JSON structure (verified against each notebook's actual save cell, not assumed), and a Langfuse wiring mechanism where this notebook's own `SIMULATED_OUTPUT` flag and each individual score's source-phase status are checked independently. A score is only ever pushed to a live dashboard as genuine when both conditions hold, not when either one alone is true.

**What was found:**

| Source Phase | Status | Headline Score |
|---|---|---|
| 02a | SIMULATED | 0.960 (RAGAS faithfulness, Gemini judge) |
| 02b | SIMULATED | 0.151 (same-family quality inflation) |
| 03a | SIMULATED | 1.000 (DeepEval RAG outcome accuracy) |
| 03b | SIMULATED | 0.857 (DeepEval governance outcome accuracy) |
| 04a | SIMULATED | 0.833 (AspectCritic pre-calibration alignment) |
| 04b | SIMULATED | 1.000 (post-calibration alignment) |
| 05a | SIMULATED | 1.000 (OWASP LLM Top 10 detection rate) |
| 05b | SIMULATED | 1.000 (OWASP Agentic Top 10 detection rate) |
| 05c | MIXED | 0.550 (ATLAS direct-match coverage) |



**The design decision that matters most in this notebook:** unlike every notebook before it, 06a's own real-mode readiness is not blocked on Gemini or Claude billing at all, it only needs Langfuse credentials, which are independent of model API costs. That decoupling is exactly why the per-source gating in Cell 6 matters: this notebook could go live today, and if it did, it would correctly refuse to push any of the current nine scores as genuine, printing an explicit skip notice for each one, rather than silently treating "this notebook is in real mode" as license to launder simulated upstream data onto a live dashboard.

**Talabat connection:** this is the exact distinction that separates a governance engineer from someone who just wires tools together, knowing that a pipeline being technically capable of running live is a different fact from the data flowing through it being real, and building the system to refuse to conflate the two rather than trusting a single flag.

**Simulated output note:** `SIMULATED_OUTPUT = True` for this notebook's own Langfuse connection. All nine upstream scores are correctly tagged with their actual source status (8 simulated, 1 mixed by design), independent of this notebook's flag.

**Next step:** Phase 6b (`06b_regression_alarm.ipynb`) builds the weekly regression alarm and production-sampled trace dataset, the last piece before Phase 7's synthesis, and will need to decide its own alerting threshold against scores that, as this phase makes explicit, are not yet real.